In [3]:
from discovery_child_development import PROJECT_DIR
import pandas as pd
import json

ENRICHED_DATA_DIR = PROJECT_DIR / 'outputs/enrichments'
PATH_TO_DATASET = ENRICHED_DATA_DIR / 'gtr_texts_relevant.csv'

# Path to topic information
PATH_TO_TOPICS = PROJECT_DIR / "discovery_child_development/pipeline/labelling/taxonomy_cat/prompts/topics.json"

# Load topic information
topics_dict = json.load(open(PATH_TO_TOPICS, 'r'))
topics = list(topics_dict.keys())
print(len(topics))

38


In [4]:
# Load the data with texts
text_df = (
    pd.read_csv(PATH_TO_DATASET)
    # .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
)
len(text_df)

1093

In [5]:
dfs = []
for topic in topics:
    keywords = topics_dict[topic]["filtering_keywords"]
    df = (
        pd.read_csv(ENRICHED_DATA_DIR / f'taxonomy_cat/gtr/taxonomy_cat_predictions_{topic}.csv')
        .merge(text_df[['id', 'text']], on='id', how='left')
    )
    keyword_hits = (
        df.text
        .str.lower()
        .str.replace(r'[^a-zA-Z0-9]', ' ', regex=True)
        .str.contains("|".join(keywords))
    )
    df = df[keyword_hits]
    dfs.append(df)

In [6]:
labelled_df = (
    pd.concat(dfs, ignore_index=True)
    # Key step: taking only data that's robustly relevant
    .query("prediction==1.0")
    .groupby("id")
    .agg(topics = ("topic", list))
    .reset_index()
)

In [7]:
labelled_text_df = (
    text_df
    .merge(labelled_df, on='id', how='left')
    .set_index("id")
)

In [9]:
# Double check specific topics
extra_keywords = {
    "ai2": ["artificial intelligence", "data science", "machine learning", "deep learning", "chatbot", "natural language processing", "computer vision", "convolutional neural network", "recurrent neural network", "reinforcement learning", "predictive model", "predictive analytics"],
    "ar_vr": ["virtual reality", "augmented reality", "mixed reality"],
    "social_media": ["social media"],
    "robotics": ["robot"],
    "parenting2": ["home learning environment", "home learning", "parenting approach", "parenting style", "home learning",
    "parenting style",
    "single parent",
    "parenting skill",
    "parenting education",
    "parenting program",
    "parenting intervention",
    "parenting support",
    "parenting practice",
    "parenting behavior",
    "parenting knowledge",
    "parenting attitude",
    "parenting guidance",
    "parenting stress",
    "parent skill",
    "parent education",
    "parent program",
    "parent intervention",
    "parent support",
    "parent practice",
    "parent behavior",
    "parent knowledge",
    "parent attitude",
    "parent guidance",
    "parent stress"],
    "wearables": ["wearable", "internet of things", " iot "],
    "mobile": ["smartphone", 'ipad', 'iphone', 'android', 'phone application'],
    "infancy": ["infant", "newborn", "neonate"],
    "protection": ["child protection", "safeguarding"],
    "communication": ["language development", "speech development"],
    "cognitive": ["cognitive development"],
    "send": ["autism", "adhd", "learning disability", "special educational needs"],
    "mental_health": [" mental health "],
    "rct": ["randomised control trial", "randomized control trial"],
    "social_services": ["social service"],
    "mobile": ['mobile phone', 'smartphone', 'android', 'iphone'],
}
extra_keywords_patents = {
    "preschool": ['preschool']
}


In [10]:
for topic in extra_keywords:
    keywords = extra_keywords[topic]
    keywords = [word.lower() for word in keywords]
    hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
    hits_ids = hits_df.id.to_list()
    labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [136]:
# for topic in extra_keywords_patents:
#     keywords = extra_keywords_patents[topic]
#     keywords = [word.lower() for word in keywords]
#     hits_df = text_df[(text_df.text.str.lower().str.replace(r'[^a-zA-Z0-9]', ' ', regex=True).str.contains("|".join(keywords)) == True)]
#     hits_ids = hits_df.query("source == 'patents'").id.to_list()
#     labelled_text_df.loc[hits_ids, "topics"] = labelled_text_df.loc[hits_ids, "topics"].apply(lambda x: list(set(x + [topic])) if isinstance(x, list) else [topic])

In [11]:
text_labelled_df = (
    pd.read_csv(PATH_TO_DATASET)
    # .assign(id = lambda df: df.id.apply(lambda x: x.split("/")[-1]))
    .merge(labelled_text_df.reset_index()[['id', 'topics']], on="id", how="left")
    .assign(topics = lambda df: df.topics.apply(lambda x: ", ".join(x) if type(x) == list else x))
    .drop(columns=["Unnamed: 0", "predictions"])
)

In [12]:
# Papers with no labels
n_with_topics = (text_labelled_df.topics.isnull() == False).sum()
n_without_topics = text_labelled_df.topics.isnull().sum()
n_with_topics / len(text_labelled_df), n_without_topics / len(text_labelled_df)

(0.918572735590119, 0.08142726440988106)

In [13]:
print(n_with_topics)

1004


In [14]:
text_labelled_df.to_csv(ENRICHED_DATA_DIR / 'taxonomy_cat/taxonomy_cat_predictions_gtr_filtered.csv', index=False)

In [16]:
text_labelled_df.sample(10)

,id,start,end,text,topics
1001,62B274E7-4902-41C9-A237-5B2E7F521970,2014-06-23,2017-06-22,EPSRC - NIHR HTC Partnership Award: Medical de...,infancy
14,23938C88-7AF2-492C-8792-25C7110C584F,2018-09-03,2024-03-31,What happens when we make art together? The im...,"emotional, infancy"
870,FD286B32-7562-4846-BD1C-BD2FB4F39973,2017-10-01,2019-03-31,Elastic jumps on networks: Quantifying pattern...,"ai2, infancy"
575,B8EB3E2F-1E9A-4DBD-B015-9A3409569BAB,2013-08-01,2016-09-30,Lexicon development in bilingual toddler. Bili...,"nontech, infancy, literacy, communication"
921,E9333806-80C9-45C7-9DDA-3BB981827E78,2018-11-01,2023-10-31,The developing human pain connectome and brain...,"infancy, prenatal"
100,C86CA2AB-AB41-4071-A6AF-839A38434265,2021-10-01,2025-03-31,Detection of Oxygen saturation (SpO2) in Newbo...,"wearables, infancy, health"
451,1A16A64B-204E-437F-B9B1-BD58ED48FEDD,2016-10-01,2021-03-31,SSA: Changes in brain circuitry caused by earl...,"neuroscience, infancy, protection, genetics, m..."
322,96380FEC-ED22-492C-B31A-658D1B829A68,2019-09-01,2024-08-31,Harnessing cross-country administrative data t...,"social_services, infancy, health, inequality"
212,628EC4EC-DD46-4634-9B39-E7DF874C9619,2019-01-01,2019-03-31,Foundations for Sound Futures. &quot;This proj...,"arts, community"
960,30735648-B205-486E-BA7F-53C2D0D932E9,2013-05-01,2015-09-30,Establishing a healthy growth trajectory from ...,"prenatal, infancy, health"


In [25]:
len(text_labelled_df)

1093

In [19]:
import pandas as pd
gtr_metadata_df = pd.read_csv(ENRICHED_DATA_DIR / 'gtr_texts.csv')

In [24]:
(
    gtr_metadata_df
    .merge(text_labelled_df[['id', 'topics']], on='id', how='inner')
).to_csv(ENRICHED_DATA_DIR / 'gtr_texts_relevant_metadata.csv', index=False)

## Extra checks